# Loading an XML Corpus

This template loads just the normalized tokens of each XML document in a folder, without token metadata, into segments based on `<s>` (i.e. sentence-like) elements. To load a verse corpus, load `<l>` (and <`lg>`) instead, as in the [_n_-grams demo notebook](https://github.com/langeslag/ehtc/blob/main/demo/n-grams.ipynb); to include metadata, adapt the template to represent each document as a list of dictionaries (cf. [parsed_corpora.ipynb](https://github.com/langeslag/ehtc/blob/main/demo/parsed_corpora.ipynb)).

In [6]:
import os
from pathlib import Path
from lxml import etree
from git import Repo

We'll ascertain the ECHOE repository has been cloned so we have XML documents to work with:

In [7]:
remote = 'https://github.com/ECHOEProject/echoe.git'
local = '../corpora/echoe'
# Only clone if the target folder doesn't already exist:
if not(Path(local).is_dir()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [8]:
# Normalization matrix:
substitutions = {
    'ę': 'æ',
    'ƿ': 'w',
    'ẏ': 'y',
    'ſ': 's',
    '': 's', # Using the glyph for descending s, instead of the unicode key point
    'v': 'u',
    'j': 'i',
    '⁊': 'and',
    ' ': '',
    '\n': ''
}

# Token normalization:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

# Discarding unwanted elements:
def simplify(branch):
    discard = ['abbr', 'am', 'sic', 'del', 'note', 'surplus']  
    # Now we define their text nodes as empty strings:
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        hit.text = ''
    return branch

In [16]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus_folder = local + '/xml/'
corpus = dict()
for file in Path(corpus_folder).glob('*.xml'):
    basename = os.path.basename(file)[:-4]
    tree = etree.parse(file, parser=parser)
    root = simplify(tree.getroot())
    segments = dict()
    for segment in root.iter('{http://www.tei-c.org/ns/1.0}s'):
        # Remember to switch to the default XML namespace to access @xml:id or @xml:lang attributes!
        identifier = segment.get('{http://www.w3.org/XML/1998/namespace}id')
        tokens = []
        for token in segment.iter('{http://www.tei-c.org/ns/1.0}w'):
            # If a word element is marked as the last part of a word, add its text content to the preceding token:
            if token.get('part') == 'F':
                position = len(tokens)-1
                tokens[position] = tokens[position] + normalize(etree.tostring(token, method='text', encoding='unicode'))
            else:
                tokens.append(normalize(etree.tostring(token, method='text', encoding='unicode')))
        segments[identifier] = tokens
    corpus[basename] = segments
        

In [17]:
corpus['018.40']['s18.40.1']

['mage',
 'we',
 'gyt',
 'her',
 'gehyran',
 'men',
 'þa',
 'leofestan',
 'eowre',
 'sawle',
 'þearfe',
 'gif',
 'ge',
 'me',
 'hlystan',
 'wyllað',
 'and',
 'on',
 'eowre',
 'heortan',
 'þas',
 'halgan',
 'lare',
 'underniman']